# Variabilidad entre corridas: comparación de modelos con múltiples semillas

Las corridas individuales del barrido inicial (`02_Preprocessing_and_Training`) muestran diferencias
pequeñas entre modelos, y además distintas ejecuciones del mismo modelo no dan exactamente los mismos
valores (variabilidad propia del entrenamiento: inicialización, orden de batches, no-determinismo de CUDA).

Para poder afirmar si una diferencia entre modelos es real o está dentro del ruido esperable, acá
repetimos el entrenamiento de un conjunto acotado de configuraciones candidatas (variantes **n**/**s** de
YOLOv8, YOLO11 y YOLO26) **5 veces cada una**, variando únicamente la semilla aleatoria. Todo lo demás
se mantiene fijo — mismas épocas, mismo batch size, mismo dataset — para que la única fuente de
variación sea el proceso estocástico del entrenamiento.

Con eso podemos reportar **media ± desvío estándar** por configuración, en lugar de un único número,
y argumentar con fundamento si las diferencias observadas entre modelos son significativas.

## Setup

In [ ]:
import os
import sys

import torch

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

try:
    from ultralytics import YOLO
except ImportError:
    print("Installing Ultralytics library...")
    !pip install ultralytics
    from ultralytics import YOLO

try:
    import wandb
except ImportError:
    print("Installing wandb...")
    !pip install wandb
    import wandb

In [ ]:
# Autenticación con wandb (mismo patrón que en 02_Preprocessing_and_Training)
# Prioridad: 1) Colab Secrets, 2) variable de entorno WANDB_API_KEY, 3) login interactivo
def get_wandb_key():
    if userdata is not None:
        try:
            return userdata.get('WandbAPIKey')
        except Exception:
            pass
    return os.environ.get('WANDB_API_KEY')

wandb_key = get_wandb_key()
if wandb_key is not None:
    wandb.login(key=wandb_key)
else:
    wandb.login(relogin=True, force=True)  # prompt interactivo

In [4]:
# Ruta al data.yaml del dataset unificado generado en 02_Preprocessing_and_Training
unified_dataset_path = "sunflower_unified"
data_yaml = os.path.join(unified_dataset_path, "data.yaml")

assert os.path.exists(data_yaml), f"No se encontró data.yaml en: {data_yaml}"

## Logging a W&B con agrupación por configuración

`with_wandb_logging` recibe ahora un parámetro `group`: un identificador común para todas las
corridas de una misma configuración (ej. `"yolo11s_sgd"`). W&B agrupa esos runs en la UI y permite
graficar **media + banda de desvío estándar** entre ellos directamente, en lugar de comparar runs
sueltos uno por uno.

También registramos la `seed` en `config` de cada run, para poder filtrarlas/identificarlas.

In [5]:
def with_wandb_logging(project, name, group=None, config=None):
    def decorator(train_fn):
        def wrapper(*args, **kwargs):
            run = wandb.init(project=project, name=name, group=group, config=config)

            model = kwargs.get("model") or args[0]

            def log_epoch(trainer):
                wandb.log({**trainer.metrics, "epoch": trainer.epoch})

            model.add_callback("on_train_epoch_end", log_epoch)

            metrics = train_fn(*args, **kwargs)

            wandb.log(metrics.results_dict)
            wandb.save(f"{project}/{name}/weights/best.pt")
            wandb.finish()

            return metrics
        return wrapper
    return decorator

## Definición de experimentos

Cada entrada de `EXPERIMENTS` define una configuración a comparar: el checkpoint base, el
optimizador y, donde corresponda, sus propios parámetros de regularización (`weight_decay`,
`dropout`, augmentations, etc. — pueden diferir entre configuraciones, ese es justamente uno de
los factores que queremos comparar).

Lo que **sí** se mantiene fijo para todas, de forma que la comparación sea limpia, son `epochs` y
`batch` (definidos como constantes comunes más abajo) y el dataset.

In [6]:
# Parámetros comunes a todas las configuraciones (para que la comparación sea justa)
COMMON_EPOCHS = 2
COMMON_BATCH = 8
COMMON_IMGSZ = 640

SEEDS = [0, 1, 2, 3, 4]  # 5 semillas por configuración

WANDB_PROJECT = "vpc2"

# Cada experimento define: checkpoint base + hiperparámetros propios de esa config.
# `extra_train_kwargs` se pasa tal cual a model.train(); ahí es donde cada config puede
# declarar su propia regularización (weight_decay, dropout, augmentations, etc.)
EXPERIMENTS = [
    {
        "group": "yolov8n_sgd",
        "checkpoint": "yolov8n.pt",
        "extra_train_kwargs": {
            "optimizer": "SGD",
            "lr0": 0.01,
            "momentum": 0.937,
            "weight_decay": 0.0005,
        },
    },
    {
        "group": "yolov8s_sgd",
        "checkpoint": "yolov8s.pt",
        "extra_train_kwargs": {
            "optimizer": "SGD",
            "lr0": 0.01,
            "momentum": 0.937,
            "weight_decay": 0.0005,
        },
    },
    {
        "group": "yolo11n_sgd",
        "checkpoint": "yolo11n.pt",
        "extra_train_kwargs": {
            "optimizer": "SGD",
            "lr0": 0.01,
            "momentum": 0.937,
            "weight_decay": 0.0005,
        },
    },
    {
        "group": "yolo11s_sgd",
        "checkpoint": "yolo11s.pt",
        "extra_train_kwargs": {
            "optimizer": "SGD",
            "lr0": 0.01,
            "momentum": 0.937,
            "weight_decay": 0.0005,
        },
    },
    {
        "group": "yolo26n_sgd",
        "checkpoint": "yolo26n.pt",
        "extra_train_kwargs": {
            "optimizer": "SGD",
            "lr0": 0.01,
            "momentum": 0.937,
            "weight_decay": 0.0005,
        },
    },
    {
        "group": "yolo26s_sgd",
        "checkpoint": "yolo26s.pt",
        "extra_train_kwargs": {
            "optimizer": "SGD",
            "lr0": 0.01,
            "momentum": 0.937,
            "weight_decay": 0.0005,
        },
    },
]

## Ejecución: una celda por configuración

Cada configuración se entrena en su propia celda (5 corridas, una por semilla en `SEEDS`),
para poder correrlas de a una, pausar entre medio, o continuar otro día sin perder lo ya
entrenado — entrenar las 6 configuraciones × 5 semillas en un solo bloque sería demasiado
largo como para ejecutarlo de una sola vez.

`run_experiment` mantiene `epochs`, `batch` e `imgsz` comunes (definidos arriba) y agrupa los
runs por `group` en W&B para poder graficar media ± desvío estándar entre semillas.

In [ ]:
device_to_use = 0 if torch.cuda.is_available() else "cpu"


def run_experiment(exp):
    """Entrena una configuración una vez por cada semilla en SEEDS, agrupando los runs en W&B."""
    group = exp["group"]

    for seed in SEEDS:
        run_name = f"{group}_seed{seed}"
        torch.cuda.empty_cache()

        model = YOLO(exp["checkpoint"])

        @with_wandb_logging(project=WANDB_PROJECT, name=run_name, group=group, config={"seed": seed, **exp["extra_train_kwargs"]})
        def train(model, **kwargs):
            return model.train(**kwargs)

        train_kwargs = {
            "model": model,
            "data": data_yaml,
            "epochs": COMMON_EPOCHS,
            "imgsz": COMMON_IMGSZ,
            "batch": COMMON_BATCH,
            "device": device_to_use,
            "workers": 2,
            "project": WANDB_PROJECT,
            "name": run_name,
            "seed": seed,
            "save": True,
            "plots": True,
            **exp["extra_train_kwargs"],
        }

        train(**train_kwargs)

        del model
        torch.cuda.empty_cache()


EXPERIMENTS_BY_GROUP = {exp["group"]: exp for exp in EXPERIMENTS}

### `yolov8n_sgd` — 5 corridas

In [ ]:
run_experiment(EXPERIMENTS_BY_GROUP["yolov8n_sgd"])

### `yolov8s_sgd` — 5 corridas

In [ ]:
run_experiment(EXPERIMENTS_BY_GROUP["yolov8s_sgd"])

### `yolo11n_sgd` — 5 corridas

In [ ]:
run_experiment(EXPERIMENTS_BY_GROUP["yolo11n_sgd"])

### `yolo11s_sgd` — 5 corridas

In [ ]:
run_experiment(EXPERIMENTS_BY_GROUP["yolo11s_sgd"])

### `yolo26n_sgd` — 5 corridas

In [ ]:
run_experiment(EXPERIMENTS_BY_GROUP["yolo26n_sgd"])

### `yolo26s_sgd` — 5 corridas

In [ ]:
run_experiment(EXPERIMENTS_BY_GROUP["yolo26s_sgd"])

## Análisis: media ± desvío estándar por configuración

Una vez completadas las corridas, se pueden traer las métricas finales de cada run desde W&B
(API `wandb.Api()`, filtrando por `group`) y calcular media y desvío estándar de `mAP50-95`,
`precision` y `recall` por configuración — esa tabla es la que sirve para argumentar en el informe
si las diferencias observadas entre modelos exceden el ruido entre semillas.

In [ ]:
import numpy as np

api = wandb.Api()


def mean_std(values):
    if not values:
        return None, None
    return float(np.mean(values)), float(np.std(values))


def fmt(mean, std):
    if mean is None:
        return "n/d"
    return f"{mean:.3f}±{std:.3f}"


summary_rows = []
for exp in EXPERIMENTS:
    group = exp["group"]
    runs = api.runs(f"{wandb.Api().default_entity}/{WANDB_PROJECT}", filters={"group": group})

    map_values = []
    precision_values = []
    recall_values = []
    for run in runs:
        summary = run.summary
        if "metrics/mAP50-95(B)" in summary:
            map_values.append(summary["metrics/mAP50-95(B)"])
        if "metrics/precision(B)" in summary:
            precision_values.append(summary["metrics/precision(B)"])
        if "metrics/recall(B)" in summary:
            recall_values.append(summary["metrics/recall(B)"])

    map_mean, map_std = mean_std(map_values)
    prec_mean, prec_std = mean_std(precision_values)
    rec_mean, rec_std = mean_std(recall_values)

    summary_rows.append({
        "group": group,
        "n_runs": len(runs),
        "mAP50-95": (map_mean, map_std),
        "precision": (prec_mean, prec_std),
        "recall": (rec_mean, rec_std),
    })

    print(f"{group:>16}  (runs={len(runs)}, mAP n={len(map_values)}, "
          f"precision n={len(precision_values)}, recall n={len(recall_values)})  "
          f"mAP50-95={fmt(map_mean, map_std)}  "
          f"precision={fmt(prec_mean, prec_std)}  "
          f"recall={fmt(rec_mean, rec_std)}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plot_rows = [r for r in summary_rows if r["mAP50-95"][0] is not None]

groups = [r["group"] for r in plot_rows]
map_means = [r["mAP50-95"][0] for r in plot_rows]
map_stds  = [r["mAP50-95"][1] for r in plot_rows]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(groups, map_means, yerr=map_stds, capsize=5, color="tab:blue", alpha=0.8)
ax.set_ylabel("mAP50-95 (media ± desvío estándar, 5 semillas)")
ax.set_title("Comparación de modelos bajo condiciones controladas (epochs y batch fijos)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()